# 🔬 IL2CPP Dumper Studio — Google Colab

A complete IL2CPP dumper for Unity Android builds, running entirely inside this
notebook.  Upload an **APK / XAPK / AAB** (or a `libil2cpp.so` + `global-metadata.dat`
pair) and get `dump.cs`, **DummyDll** assemblies, `il2cpp.h`, `script.json` and
string literals.

**Developer: Mohamed Annati** · Nothing leaves this notebook — every file is
processed locally in the Colab VM.

---

### How to use
1. Press the ▶️ button on each cell in order.
2. Cell **2** starts the web studio and prints a `https://...e2b.app` / Colab proxy
   link — open it.
3. In the web UI, drop your APK and press **Dump IL2CPP**.

Prefer the command line? Use **Cell 3** instead.

In [ ]:
#@title 1 · Install dependencies  📦
import os, sys

REPO = "IL2CPP-Dumper-Studio"
GITHUB_USER = "Mohamed2020p"
BRANCH = "arena/01a04eae-mainproject"

if not os.path.isdir(REPO):
    !git clone --depth 1 -b {BRANCH} https://github.com/{GITHUB_USER}/mainproject {REPO}
else:
    !git -C {REPO} pull --ff-only 2>/dev/null || True

sys.path.insert(0, os.path.abspath(REPO))
os.chdir(REPO)

# Flask powers the web UI.  Pillow is optional (only for regenerating brand PNGs).
try:
    import flask
except ImportError:
    !pip install -q flask
    import flask

print("\u2705 Ready.  Flask", flask.__version__ if hasattr(flask, "__version__") else "")

In [ ]:
#@title 2 · Launch the web studio  🌐
# Opens a live web UI through the Colab reverse proxy.
from google.colab import output as _colab_output
import threading
from app import server as _server

PORT = 8050

def _run():
    _server.APP.run(host="0.0.0.0", port=PORT, debug=False, threaded=True, use_reloader=False)

threading.Thread(target=_run, daemon=True).start()

proxy = _colab_output.eval_js("google.colab.kernel.proxyPort(%d, {'cache': false})" % PORT)
print("\n🔗  Open the web studio:\n\n     " + proxy + "\n")

In [ ]:
#@title 3 · Command-line dumper (alternative)  💻
# Upload your files with the folder icon on the left, then edit the paths below.
from dumper.cli import main as _cli_main

APK_PATH     = "/content/game.apk"        #@param {type:"string"}
OUTPUT_DIR   = "/content/dump"            #@param {type:"string"}

# The CLI auto-detects whether it got an APK or a .so/.dat pair.
_cli_main([APK_PATH, "-o", OUTPUT_DIR])

print("\n📁 Results written to", OUTPUT_DIR)

### Downloading your results

The web UI offers one-click downloads for every artefact and a **Download everything
(.zip)** button.  From the command line, zip the folder yourself:

```python
!cd /content && zip -r dump.zip dump
from google.colab import files; files.download("/content/dump.zip")
```

### What each output is for
| File | Purpose |
|---|---|
| `dump.cs` | C# pseudo-code of every type (methods, fields, properties, RVAs) |
| `DummyDll/` | Rebuilt .NET assemblies to open in **dnSpy / ILSpy** |
| `il2cpp.h` | Runtime structures for IDA / Ghidra |
| `script.json` | Symbol + string table to rename everything in IDA / Ghidra |
| `stringliteral.json` | Every managed string literal |

---
*For educational reverse-engineering of software you own or are licensed to analyse.*